# 02 — Eksperimen Baseline: Kurs-saja vs Kurs+Berita

Tugas: **klasifikasi biner NAIK/TURUN** (`direction`).
Split kronologis 70/15/15 (train 826 / val 177 / test 178 hari).
Metrik utama: **accuracy (= directional accuracy)**, F1, F1-macro, AUC.
Jalankan dari folder `notebook/` setelah `build_dataset` + kedua skrip training.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

base = json.load(open('../outputs/results/baseline_metrics.json'))
comb = json.load(open('../outputs/results/combined_metrics.json'))
pd.DataFrame(base).T[['accuracy', 'f1', 'f1_macro', 'auc']].round(3)

## Hasil baseline kurs-saja (precomputed)

| Model | Val acc / F1-macro / AUC | Test acc / F1-macro / AUC |
|---|---|---|
| Naive mayoritas | 0.520 / 0.342 / – | 0.579 / 0.367 / – |
| Naive persistensi | 0.514 / 0.513 / – | 0.545 / 0.533 / – |
| LogReg | 0.531 / 0.494 / 0.507 | 0.522 / 0.520 / 0.548 |
| GradBoost | 0.497 / 0.495 / 0.534 | 0.483 / 0.458 / 0.504 |

Bacaan: model kurs-saja ≈ acak (0,48–0,53). Persistensi sedikit lebih baik dari mayoritas dalam F1-macro — ada sedikit momentum harian, tetapi lemah.

## Hasil gabungan — ablation fitur NLP (test, precomputed)

| Varian (GradBoost / LogReg) | Test acc | Test F1-macro | Test AUC |
|---|---|---|---|
| tech_only | 0.483 / 0.522 | 0.458 / 0.520 | 0.504 / 0.548 |
| +lex_title (VADER+LM judul) | **0.590** / **0.601** | **0.579** / 0.552 | 0.572 / 0.596 |
| +lex_full (VADER+LM isi) | 0.511 / 0.584 | 0.492 / 0.458 | 0.529 / 0.610 |
| +lex_all | 0.461 / 0.579 | 0.451 / 0.482 | 0.522 / 0.615 |
| +tfidf_only | 0.601 / 0.573 | 0.535 / 0.522 | 0.590 / 0.543 |
| +all | 0.601 / 0.579 | 0.503 / 0.401 | 0.556 / 0.612 |

Varian terbaik di **val** (`+lex_title/gradboost`, F1-macro 0,496) → di test: acc 0,590 / F1-macro 0,579 / AUC 0,572, confusion `[[38, 37], [36, 67]]` (n=178).

In [ ]:
# Grafik: akurasi test per varian (GradBoost)
rows = [(k.rsplit('@', 1)[0].rsplit('/', 1)[0], v['accuracy'])
        for k, v in comb.items() if k.endswith('@test') and '/gradboost' in k]
df = pd.DataFrame(rows, columns=['varian', 'acc']).sort_values('acc')
plt.figure(figsize=(7, 4))
plt.barh(df['varian'], df['acc'])
plt.axvline(0.5, linestyle='--')
plt.title('Akurasi TEST per varian fitur (GradBoost)')
plt.xlabel('accuracy')
plt.tight_layout()
plt.show()

## Analisis ablation: judul vs isi

1. **Judul > isi, konsisten di kedua learner.** `+lex_title` menambah +6–8 pp akurasi test atas `tech_only`; `+lex_full` hanya +1–3 pp (GradBoost) dengan F1-macro lebih rendah. Hipotesis: judul padat sinyal, isi penuh noise dan topik tak relevan.
2. **Menggabung judul+isi (`+lex_all`) tidak membantu** — akurasi GradBoost malah turun ke 0,461. Redundansi + dimensi.
3. **TF-IDF membantu akurasi tetapi F1-macro biasa saja** (0,535/0,522): menangkap kata kunci rezim, bukan sentimen seimbang.
4. **Peringatan validitas**: semua varian ≈ 0,50 di val tetapi 0,51–0,60 di test. Test (Nov 2025–Agu 2026, NAIK 57,9%) tampaknya rezim berbeda (tren melemah kuat, liputan tarif/AS). Klaim “NLP membantu” butuh walk-forward validation + uji signifikansi sebelum dipercaya.

## Next step yang disarankan

1. Walk-forward / purged time-series CV (ganti single split).
2. Filter topik: hanya artikel bertema Fed/BI/trade/FX sebelum vectorize.
3. Kalibrasi ambang probabilitas + laporkan *trading-relevant* metric (mis. return strategi).
4. Uji McNemar (`+lex_title` vs `tech_only`) dan learning curve vs jumlah berita.
5. Ekstensi regresi (`regression_metrics` sudah tersedia): prediksi `kurs_next`, evaluasi RMSE/MAE.